# 08. Настоящий VAE: ручной протокол PyTorch и ComfyUI

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Подготовим измеримый pretrained эксперимент без автоматической загрузки весов или запуска ComfyUI. Сейчас выполняется только dry-run плана.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT итог:** Part 4 VAE и Part 5 latent DiT → промышленный frozen codec. ComfyUI показывает reconstruction; он не вычисляет ELBO и не заменяет реализацию Q4.7. [Полный маршрут](../MIT_LAB3_BRIDGE.md).


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='08_pretrained_and_comfyui_protocol';metrics={}
print('Pinned source:',source_hash())


## 1. Исследовательская карточка

**Вопрос:** при заданном budget какой spatial/temporal tile даёт минимальный peak memory, сохраняя размер/длительность и допустимую reconstruction quality?

**Материалы:** 3–5 локальных коротких clips с согласием на использование: static texture/text, motion, scene transition. Не нужно качать DiT, Qwen, CLIP, SR, audio VAE: только соответствующий video VAE. Tiny experiments уже есть в 01–07; pretrained checkpoint должен быть идентифицирован SHA/config/provenance отдельно.

**Порядок:** decode video→RGB float [-1,1]→BCTHW→pad/alignment с valid mask→posterior.mode→decode→unpad→metrics. Сначала F=1, затем 17 и 33; 32×32 / 64×64 для первого shape smoke, затем 240×432 (H кратна 8). Для F=33/17 допустима grid из предыдущих labs, но конкретный planner всё равно проверяем.

**Контроль:** один checkpoint, dtype/device, input, posterior.mode, weights и warmup. Начинайте с full inference на малом input, затем spatial, temporal, combined. Масштабируйте input только после корректного full baseline. Если full на большом input OOM, quality baseline берите на меньшем clip; не выдавайте другой shape за paired full comparison.

**Ограничение станции:** pretrained phase отложен пользователем до освобождения 50 GB+ SSD. Этот notebook не проверяет gate вместо человека и не запускает загрузку. MPS support требует отдельного smoke; CUDA free-memory heuristic в attached code для MPS не действует.


In [ ]:
import subprocess
runner=COURSE/'scripts/benchmark_local_codec.py'
p=subprocess.run([sys.executable,str(runner),'--help'],capture_output=True,text=True,check=True)
print(p.stdout)
plan_record={'execute':False,'codec':'local Diffusers-format Hunyuan VAE directory',
 'input':'local BCTHW float tensor saved as .npy, RGB [-1,1]',
 'policies':['full','spatial','temporal','combined'],
 'device':'cpu for smoke / mps experimental / cuda for measured VRAM',
 'no_network':True,'no_weights_loaded':True}
metrics['manual_plan']=plan_record
print(json.dumps(plan_record,ensure_ascii=False,indent=2))


## 2. Команда ручного запуска после resource gate

Runner использует установленный **Diffusers HunyuanVideo VAE** и local_files_only=True. Это отдельная reference реализация; он **не утверждает**, что её tiling совпадает с attached adaptation. Версия, class и VAE config сохраняются. При недоступном требуемом flag runner останавливается, а не делает silent fallback.

```bash
# Из корня репозитория, в отдельно подготовленном пользователем окружении с Diffusers.
# Сначала dry-run: не загружает веса и не импортирует Diffusers.
python vae/scripts/benchmark_local_codec.py \
  --model /ABS/LOCAL/vae --input /ABS/LOCAL/clip.npy --device mps
# После проверки путей/места добавьте --execute, например для одной policy:
python vae/scripts/benchmark_local_codec.py \
  --model /ABS/LOCAL/vae --input /ABS/LOCAL/clip.npy --device cuda \
  --dtype float16 --policy spatial --repeats 3 --execute
```

Нужен Diffusers-format folder с config.json и весами. ComfyUI single `.safetensors` не следует просто переименовывать: architecture/config/key mapping проверяются отдельно. Не устанавливайте пакеты в Desktop .venv ради этого notebook. Adapter к attached native class — следующий отдельный runtime experiment после проверки import/weight format/API; source-level code в 00–07 уже изучен.

Runner рассчитывает PSNR/MSE на floating RGB, **до MP4 compression**, запускает deterministic mode, синхронизирует device вокруг timing. На CUDA сбрасывает peak counters после загрузки модели и warmup; peak absolute allocated включает резидентные веса, incremental peak считается относительно baseline. Reserved показывается отдельно. На MPS записывает before/after allocation counters: **это не peak**. RSS не называем VRAM.


## 3. ComfyUI как визуальный контроль

Откройте существующий [workflow codec reconstruction](../../curriculum/workflows/04_hunyuan_vae_reconstruction.json), инструкция [C4](../../curriculum/COMFYUI_LABS.md). Он работает с image F=1: позволяет увидеть normalization/color/tiling artifacts без DiT. Для temporal опыта нужен nodes graph, который сохраняет **целый video batch** и вызывает temporal VAE interface; серия независимых image encode не проверяет causal video codec.

1. Загрузите [test chart](../../curriculum/assets/curriculum_test_chart.png), выберите **уже имеющийся** Hunyuan-compatible VAE. Зафиксируйте ComfyUI revision/custom nodes и SHA checkpoint. Откажитесь от автоматической догрузки ненужных pipeline моделей.
2. Запустите decode(encode(image)); сохраните исходный PNG и reconstruction PNG. Сначала обычные VAEEncode/VAEDecode; tiled nodes добавляйте только при наличии в вашей версии. Запишите реальные tile/overlap settings.
3. Очистите/обойдите кэш для измерения: неизменённый graph может не вызвать VAE повторно. Сначала warmup, затем 3 фактических repetitions; следите, что nodes исполнялись.
4. Сравните через [compare_images.py](../../curriculum/scripts/compare_images.py), не screenshot preview. Учтите RGB/sRGB, clipping и PNG quantization: это другой comparison stage, чем float tensors runner.
5. Для видео сначала проверяйте F=17/33, FPS и число output frames. MP4 используйте для просмотра после численных измерений; отдельно экспортируйте float/PNG frames для точного сравнения.

ComfyUI обычно использует собственный loader/wrapper. Результат графа не является автоматически runtime test именно attached `get_enc_optimal_tiling`. Зафиксируйте исполненный class/source revision, а не только имя nodes.


In [ ]:
# Template only: no HTTP calls to Desktop and no queued generation.
columns=['run_id','implementation','revision','checkpoint_sha256','input_sha256','device','dtype',
 'F','H','W','policy','tile_F','tile_H','tile_W','stride_F','stride_H','stride_W',
 'posterior','warmup','repeats','seconds_median','peak_allocated_bytes','peak_reserved_bytes',
 'mps_current_before','mps_current_after','mse_input','psnr_input','mse_full',
 'seam_mse','interior_mse','output_F','output_H','output_W','status','notes']
metrics['measurement_columns']=columns
print(','.join(columns))


## 4. Как сделать вывод, когда появятся реальные числа

- **Корректность:** все lengths/layout, отсутствие NaN и aligned coverage. OOM — отдельный статус, не качество=0.
- **Качество:** paired MSE/PSNR to full и input; seam/interior; temporal consistency на motion clips. Не сравнивайте только один static image.
- **Память:** CUDA peak allocated и reserved в bytes; MPS counters before/after не дают пика. Измерение должно указывать implementation, shape, dtype и policy.
- **Скорость:** warmup, синхронизация, median и разброс, отдельно encode и decode. ComfyUI cache/download/load times не смешиваем с kernel inference.
- **Ограничения:** small clips и fixed hardware; результат не распространяется автоматически на 1080p/длинное видео или другой checkpoint.

Для честного сравнения adaptive vs fixed tiling нужны оба runtime на одних данных и budgets. Source header обещает lower peak относительно Diffusers, но наша CPU серия это **не проверяет**. Исследовательский вывод пока: algebra/indexing/runtime interfaces изучены; pretrained resource/quality evidence ожидает ручного запуска.


## Упражнения и критерий перехода

1. Заполните один dry-run record до покупки GPU.
2. Выберите допустимую degradation threshold и memory reserve **до** просмотра результатов.
3. После запуска сравните 3–5 clips и сформулируйте минимальный поддержанный вывод. Критерий завершения: таблица real runs с provenance, корректными units и отдельно описанными failures.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
